# Ribonanza split + tokenization

Goal: a reproducible 90/5/5 train/val/test split of the Ribonanza pretraining data, plus tokenized sequences.

1. Load only the metadata columns (no reactivities needed to split)
2. Keep high signal-to-noise rows and make sure each sequence has both DMS and 2A3
3. Remove sequences that overlap OpenVaccine held-out data
4. Split by **unique sequence**, so DMS/2A3 rows and any duplicated sequences always land in the same split
5. Save the split assignments (and optionally full-schema CSVs)
6. Tokenize sequences and save as `.npy`

Everyone on the team gets identical splits as long as they use the same `SEED` and input file.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

INPUT_CSV = Path('train_data_QUICK_START.csv')
OUTPUT_DIR = Path('preprocessed_ribonanza')
SEED = 2026
MAX_LENGTH = 206
MAX_SEQUENCES = None            # e.g. 50_000 to subsample unique sequences before splitting
VAL_FRAC, TEST_FRAC = 0.05, 0.05  # train gets the remainder
WRITE_SPLIT_CSVS = True         # also write train/val/test CSVs in the original schema (with reactivities)

# OpenVaccine val/test files for the leakage check (.json lines or .csv with a 'sequence' column)
OPENVACCINE_HELDOUT_FILES = [Path('private_test.json')]  # e.g. [Path('openvaccine/test.json')]
KMER = 40                       # a shared stretch this long counts as overlap

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and inspect

Only metadata columns are loaded, so this is fast and light on memory.

In [2]:
header = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
meta_cols = [c for c in ['sequence_id', 'sequence', 'experiment_type', 'signal_to_noise', 'SN_filter'] if c in header]
df = pd.read_csv(INPUT_CSV, usecols=meta_cols,
                 dtype={'sequence_id': 'string', 'sequence': 'string', 'experiment_type': 'string'})

print('Shape:', df.shape, '| columns:', meta_cols)
print(df['experiment_type'].value_counts(), '\n')

lengths = df['sequence'].str.len()
print(lengths.describe(), '\n')
assert df['sequence'].str.fullmatch('[ACGU]+').all(), 'Unexpected nucleotide symbols'
assert lengths.max() <= MAX_LENGTH, 'Sequence longer than MAX_LENGTH'

Shape: (335616, 3) | columns: ['sequence_id', 'sequence', 'experiment_type']
experiment_type
2A3_MaP    167808
DMS_MaP    167808
Name: count, dtype: int64[pyarrow] 

count      335616.0
mean     175.652037
std        8.854856
min           115.0
25%           177.0
50%           177.0
75%           177.0
max           206.0
Name: sequence, dtype: Float64 



## 2. Pair the assays by sequence id

The quick-start file may already be filtered, so this reports what it finds rather than assuming. Filtering can drop one assay of a pair, so pairing is checked *after* filtering.

In [5]:
# keep only sequence_ids that have both assays
n_assays = df.groupby('sequence_id')['experiment_type'].nunique()
paired_ids = n_assays.index[n_assays == 2]
print(f'Dropping {(n_assays < 2).sum():,} IDs missing an assay; keeping {len(paired_ids):,}')
df = df[df['sequence_id'].isin(paired_ids)]

assert df.groupby('sequence_id')['sequence'].nunique().eq(1).all(), 'An ID maps to multiple sequences'

# one row per sequence_id
seqs = (df.drop_duplicates('sequence_id')[['sequence_id', 'sequence']]
          .sort_values('sequence_id').reset_index(drop=True))
print(f"{len(seqs):,} paired IDs, {seqs['sequence'].nunique():,} unique sequences")

Dropping 0 IDs missing an assay; keeping 167,808
167,808 paired IDs, 167,808 unique sequences


## 3. Leakage check against OpenVaccine

Flags any Ribonanza sequence that shares a stretch of `KMER` nucleotides with an OpenVaccine held-out sequence (this also catches exact matches and containment). Flagged sequences are removed before splitting. If a large fraction gets flagged, something like shared constant flanking regions is probably the cause, so the cell stops and lets you inspect instead of silently deleting data.

In [6]:
def kmers(seq, k=KMER):
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}

heldout = set()
for path in OPENVACCINE_HELDOUT_FILES:
    ov = pd.read_json(path, lines=True) if Path(path).suffix == '.json' else pd.read_csv(path)
    heldout |= set(ov['sequence'])

leakage_checked = len(heldout) > 0
n_leaky_removed = 0
if leakage_checked:
    heldout_kmers = set().union(*(kmers(s) for s in heldout))
    flagged = seqs['sequence'].map(lambda s: not heldout_kmers.isdisjoint(kmers(s)))
    exact = int(seqs['sequence'].isin(heldout).sum())
    print(f'{len(heldout):,} held-out sequences loaded; exact matches: {exact}; '
          f'sharing a {KMER}-mer: {int(flagged.sum())} ({flagged.mean():.2%})')
    if flagged.mean() > 0.05:
        raise RuntimeError('Over 5% of sequences flagged. Inspect the overlap (shared flanks?) before dropping.')
    n_leaky_removed = int(flagged.sum())
    seqs = seqs[~flagged].reset_index(drop=True)
else:
    print('WARNING: no OpenVaccine files configured. Leakage check NOT run.')

3,005 held-out sequences loaded; exact matches: 0; sharing a 40-mer: 513 (0.31%)


## 4. Split by unique sequence

Unique sequences are sorted, shuffled with the seed, and assigned to splits. Every ID sharing a sequence inherits that sequence's split.

In [7]:
unique_seqs = np.array(sorted(seqs['sequence'].unique()))
rng = np.random.default_rng(SEED)
unique_seqs = unique_seqs[rng.permutation(len(unique_seqs))]
if MAX_SEQUENCES is not None:
    unique_seqs = unique_seqs[:MAX_SEQUENCES]

n = len(unique_seqs)
n_val, n_test = max(1, round(VAL_FRAC * n)), max(1, round(TEST_FRAC * n))
assert n - n_val - n_test > 0, 'Subset too small to split'

split_of_seq = pd.Series('train', index=unique_seqs)
split_of_seq.iloc[:n_val] = 'val'
split_of_seq.iloc[n_val:n_val + n_test] = 'test'

seqs = seqs[seqs['sequence'].isin(split_of_seq.index)].copy()
seqs['split'] = seqs['sequence'].map(split_of_seq)
print(seqs['split'].value_counts())

split
train    150565
test       8365
val        8365
Name: count, dtype: int64


## 5. Save split assignments

`splits.csv` is the source of truth: one row per `sequence_id` with its split. The optional full CSVs are streamed from the original file so they keep every column, including reactivities.

In [8]:
seqs[['sequence_id', 'sequence', 'split']].to_csv(OUTPUT_DIR / 'splits.csv', index=False)

if WRITE_SPLIT_CSVS:
    split_of_id = dict(zip(seqs['sequence_id'], seqs['split']))
    started, rows_written = set(), {'train': 0, 'val': 0, 'test': 0}
    for chunk in pd.read_csv(INPUT_CSV, chunksize=10_000, low_memory=False):
        chunk_split = chunk['sequence_id'].astype(str).map(split_of_id)
        for split in rows_written:
            part = chunk[chunk_split == split]
            if part.empty:
                continue
            part.to_csv(OUTPUT_DIR / f'{split}.csv', index=False,
                        mode='a' if split in started else 'w', header=split not in started)
            started.add(split)
            rows_written[split] += len(part)
    for split, count in seqs['split'].value_counts().items():
        assert rows_written[split] == 2 * count, f'{split}: {rows_written[split]} rows, expected {2 * count}'
    print('Rows written:', rows_written)

Rows written: {'train': 301130, 'val': 16730, 'test': 16730}


## 6. Tokenize

Each nucleotide becomes an integer: `A=1, C=2, G=3, U=4`, and `0` is padding out to length 206. Neural networks can't read letters, so this is the form an embedding layer (or one-hot encoder) takes as input. Position `j` of row `i` is nucleotide `j` of sequence `i`.

Rows in each saved array are ordered by `sequence_id`, so `{split}_sequence_ids.npy` tells you which row is which.

In [9]:
TOKEN_MAP = {'PAD': 0, 'A': 1, 'C': 2, 'G': 3, 'U': 4}
lookup = np.zeros(256, dtype=np.uint8)
for base in 'ACGU':
    lookup[ord(base)] = TOKEN_MAP[base]

def tokenize(sequences):
    out = np.zeros((len(sequences), MAX_LENGTH), dtype=np.uint8)
    for i, s in enumerate(sequences):
        out[i, :len(s)] = lookup[np.frombuffer(s.encode('ascii'), dtype=np.uint8)]
    return out

for split in ['train', 'val', 'test']:
    part = seqs[seqs['split'] == split].sort_values('sequence_id')
    tokens = tokenize(part['sequence'].tolist())
    np.save(OUTPUT_DIR / f'{split}_tokens.npy', tokens, allow_pickle=False)
    np.save(OUTPUT_DIR / f'{split}_lengths.npy', part['sequence'].str.len().to_numpy(np.int16), allow_pickle=False)
    np.save(OUTPUT_DIR / f'{split}_sequence_ids.npy', part['sequence_id'].to_numpy(dtype=str), allow_pickle=False)
    print(split, tokens.shape, tokens.dtype)

train (150565, 206) uint8
val (8365, 206) uint8
test (8365, 206) uint8


## 7. Sanity checks and metadata

In [10]:
# no overlap between splits, by ID or by sequence
by_split = {s: seqs[seqs['split'] == s] for s in ['train', 'val', 'test']}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not set(by_split[a]['sequence_id']) & set(by_split[b]['sequence_id'])
    assert not set(by_split[a]['sequence']) & set(by_split[b]['sequence'])

# tokens decode back to the original sequences
inverse = {v: k for k, v in TOKEN_MAP.items() if k != 'PAD'}
for split, part in by_split.items():
    part = part.sort_values('sequence_id')
    tokens = np.load(OUTPUT_DIR / f'{split}_tokens.npy')
    lens = np.load(OUTPUT_DIR / f'{split}_lengths.npy')
    ids = np.load(OUTPUT_DIR / f'{split}_sequence_ids.npy')
    assert (ids == part['sequence_id'].to_numpy(dtype=str)).all()
    for i in range(min(200, len(part))):
        decoded = ''.join(inverse[t] for t in tokens[i, :lens[i]])
        assert decoded == part['sequence'].iloc[i]
        assert (tokens[i, lens[i]:] == 0).all()
print('Checks passed: no split overlap, tokens round-trip, padding correct.')

metadata = {
    'input_csv': str(INPUT_CSV), 'seed': SEED, 'max_sequences': MAX_SEQUENCES,
    'max_length': MAX_LENGTH, 'token_map': TOKEN_MAP,
    'sn_filter_applied': 'SN_filter' in header,
    'split_counts_ids': {s: int(len(p)) for s, p in by_split.items()},
    'split_counts_unique_sequences': {s: int(p['sequence'].nunique()) for s, p in by_split.items()},
    'openvaccine_leakage_checked': leakage_checked,
    'openvaccine_kmer': KMER if leakage_checked else None,
    'sequences_removed_for_leakage': n_leaky_removed,
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

Checks passed: no split overlap, tokens round-trip, padding correct.
{
  "input_csv": "train_data_QUICK_START.csv",
  "seed": 2026,
  "max_sequences": null,
  "max_length": 206,
  "token_map": {
    "PAD": 0,
    "A": 1,
    "C": 2,
    "G": 3,
    "U": 4
  },
  "sn_filter_applied": false,
  "split_counts_ids": {
    "train": 150565,
    "val": 8365,
    "test": 8365
  },
  "split_counts_unique_sequences": {
    "train": 150565,
    "val": 8365,
    "test": 8365
  },
  "openvaccine_leakage_checked": true,
  "openvaccine_kmer": 40,
  "sequences_removed_for_leakage": 513
}
